# Human Pose Estimation với MoveNet Singlepose Thunder

**Quy trình 2 bước:**
1. `extract_keypoints()`: Trích xuất keypoints từ video → lưu file
2. `render_video()`: Parse keypoints và vẽ lên video

**Output tensor format:** `(T, 17, [x, y, c])` - T frames, 17 keypoints, mỗi keypoint có [x, y, confidence]

## 1. Import Libraries

In [1]:
import cv2
import json
import numpy as np 
import tensorflow as tf 
import tensorflow_hub as hub
from tqdm.notebook import tqdm
from collections import deque

d:\PBL5\venv\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


## 2. Configuration

In [2]:
# Model: Singlepose Thunder (chính xác nhất)
MODEL_URL = "https://tfhub.dev/google/movenet/singlepose/thunder/4"
MODEL_INPUT_SIZE = 256  # Singlepose Thunder: 256x256

# Màu sắc cho các edges (BGR format cho OpenCV)
cyan = (255, 255, 0)
magenta = (255, 0, 255)
green = (0, 255, 0)

# Edge connections
EDGE_COLORS = {
    (0, 1): magenta,   # nose -> left_eye
    (0, 2): cyan,      # nose -> right_eye
    (1, 3): magenta,   # left_eye -> left_ear
    (2, 4): cyan,      # right_eye -> right_ear
    (0, 5): magenta,   # nose -> left_shoulder
    (0, 6): cyan,      # nose -> right_shoulder
    (5, 7): magenta,   # left_shoulder -> left_elbow
    (7, 9): cyan,      # left_elbow -> left_wrist
    (6, 8): magenta,   # right_shoulder -> right_elbow
    (8, 10): cyan,     # right_elbow -> right_wrist
    (5, 6): magenta,   # left_shoulder -> right_shoulder
    (5, 11): cyan,     # left_shoulder -> left_hip
    (6, 12): magenta,  # right_shoulder -> right_hip
    (11, 12): cyan,    # left_hip -> right_hip
    (11, 13): magenta, # left_hip -> left_knee
    (13, 15): cyan,    # left_knee -> left_ankle
    (12, 14): magenta, # right_hip -> right_knee
    (14, 16): cyan     # right_knee -> right_ankle
}

# 17 Keypoints của MoveNet
KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

## 3. Load Model

In [3]:
print("=" * 50)
print("Loading MoveNet Singlepose Thunder model...")
print("=" * 50)

WIDTH = HEIGHT = MODEL_INPUT_SIZE

model = hub.load(MODEL_URL)
movenet = model.signatures["serving_default"]

print("✅ Model loaded: Singlepose Thunder")

Loading MoveNet Singlepose Thunder model...



✅ Model loaded: Singlepose Thunder


## 4. Smoothing Class

In [4]:
class KeypointSmoother:
    """
    Temporal smoothing cho keypoints sử dụng exponential moving average
    Giúp giảm nhiễu và làm mượt chuyển động
    """
    
    def __init__(self, smoothing_factor=0.5, window_size=5):
        """
        Args:
            smoothing_factor: 0-1, càng cao càng mượt nhưng càng lag
            window_size: Số frames để tính moving average
        """
        self.smoothing_factor = smoothing_factor
        self.window_size = window_size
        self.history = deque(maxlen=window_size)
        
    def smooth(self, keypoints):
        """
        Smooth keypoints
        
        Args:
            keypoints: Array [17, 3] chứa keypoints với format (y, x, confidence)
        
        Returns:
            smoothed_keypoints: Array [17, 3] đã được làm mượt
        """
        self.history.append(keypoints.copy())
        
        if len(self.history) < 2:
            return keypoints
        
        # Exponential moving average
        smoothed = np.zeros_like(keypoints)
        history = list(self.history)
        
        for i in range(17):
            # Chỉ smooth nếu confidence của keypoint hiện tại > threshold
            if keypoints[i, 2] > 0.1:
                weights = []
                values_y = []
                values_x = []
                
                for j, hist_kp in enumerate(history):
                    if hist_kp[i, 2] > 0.1:  # Chỉ xét keypoints có confidence
                        weight = (j + 1) / len(history)  # Frame mới hơn có weight cao hơn
                        weights.append(weight * hist_kp[i, 2])  # Nhân với confidence
                        values_y.append(hist_kp[i, 0])
                        values_x.append(hist_kp[i, 1])
                
                if weights:
                    total_weight = sum(weights)
                    smoothed[i, 0] = sum(w * v for w, v in zip(weights, values_y)) / total_weight
                    smoothed[i, 1] = sum(w * v for w, v in zip(weights, values_x)) / total_weight
                    smoothed[i, 2] = keypoints[i, 2]  # Giữ confidence hiện tại
                else:
                    smoothed[i] = keypoints[i]
            else:
                smoothed[i] = keypoints[i]
        
        return smoothed
    
    def reset(self):
        """Reset lịch sử"""
        self.history.clear()

## 5. Helper Functions (Improved Validation)

In [5]:
def is_valid_skeleton(keypoints, threshold=0.3, min_keypoints=6):
    """
    Kiểm tra xem skeleton có hợp lệ không
    
    Validation logic:
    1. Đếm số keypoints có confidence > threshold
    2. Kiểm tra core keypoints (shoulders, hips) có đủ không
    3. Kiểm tra tỷ lệ cơ thể có hợp lý không
    
    Args:
        keypoints: Array [17, 3] với format (y, x, confidence)
        threshold: Ngưỡng confidence
        min_keypoints: Số keypoints tối thiểu phải đạt threshold
    
    Returns:
        bool: True nếu skeleton hợp lệ
    """
    # Đếm số keypoints valid
    valid_count = np.sum(keypoints[:, 2] > threshold)
    
    if valid_count < min_keypoints:
        return False
    
    # Kiểm tra các keypoints quan trọng (core body)
    # Index: 5,6 = shoulders, 11,12 = hips
    core_indices = [5, 6, 11, 12]  # shoulders và hips
    core_valid = sum(1 for i in core_indices if keypoints[i, 2] > threshold)
    
    # Ít nhất 2 trong 4 core keypoints phải valid
    if core_valid < 2:
        return False
    
    # Kiểm tra tỷ lệ cơ thể (body proportion check)
    # Khoảng cách shoulders không nên quá nhỏ so với khoảng cách shoulder-hip
    if keypoints[5, 2] > threshold and keypoints[6, 2] > threshold:
        shoulder_dist = np.sqrt((keypoints[5, 0] - keypoints[6, 0])**2 + 
                                (keypoints[5, 1] - keypoints[6, 1])**2)
        
        # Nếu có hip keypoints, kiểm tra tỷ lệ
        if keypoints[11, 2] > threshold or keypoints[12, 2] > threshold:
            hip_idx = 11 if keypoints[11, 2] > keypoints[12, 2] else 12
            shoulder_idx = 5  # left shoulder
            torso_dist = np.sqrt((keypoints[shoulder_idx, 0] - keypoints[hip_idx, 0])**2 + 
                                 (keypoints[shoulder_idx, 1] - keypoints[hip_idx, 1])**2)
            
            # Tỷ lệ bất thường: shoulders quá rộng hoặc torso quá ngắn
            if torso_dist > 0 and shoulder_dist / torso_dist > 2.0:
                return False
    
    return True


def filter_outlier_keypoints(keypoints, threshold=0.3):
    """
    Lọc các keypoints bất thường (outliers)
    
    Logic:
    1. Kiểm tra khoảng cách giữa các cặp keypoints đối xứng (vai-vai, hông-hông...)
    2. Kiểm tra khoảng cách giữa các keypoints liên kết (vai-khuỷu, khuỷu-cổ tay...)
    3. Kiểm tra keypoints có nằm trong frame không
    
    Args:
        keypoints: Array [17, 3] với format (y, x, confidence)
        threshold: Ngưỡng confidence
    
    Returns:
        filtered: Keypoints đã lọc outliers
    """
    filtered = keypoints.copy()
    
    # Các cặp keypoints đối xứng và max distance hợp lý (normalized 0-1)
    symmetric_pairs = [
        (5, 6, 0.4),    # shoulders - không nên quá xa
        (7, 8, 0.5),    # elbows
        (9, 10, 0.6),   # wrists - có thể xa hơn (giơ tay)
        (11, 12, 0.35), # hips - thường gần nhau
        (13, 14, 0.5),  # knees
        (15, 16, 0.6),  # ankles
    ]
    
    # Các cặp keypoints liên kết và max distance
    connected_pairs = [
        (5, 7, 0.35),   # shoulder -> elbow
        (7, 9, 0.35),   # elbow -> wrist
        (6, 8, 0.35),   # shoulder -> elbow
        (8, 10, 0.35),  # elbow -> wrist
        (5, 11, 0.5),   # shoulder -> hip (torso)
        (6, 12, 0.5),   # shoulder -> hip (torso)
        (11, 13, 0.4),  # hip -> knee
        (13, 15, 0.4),  # knee -> ankle
        (12, 14, 0.4),  # hip -> knee
        (14, 16, 0.4),  # knee -> ankle
    ]
    
    all_pairs = [(p1, p2, max_d, True) for p1, p2, max_d in symmetric_pairs] + \
                [(p1, p2, max_d, False) for p1, p2, max_d in connected_pairs]
    
    for p1, p2, max_dist, is_symmetric in all_pairs:
        if filtered[p1, 2] > threshold and filtered[p2, 2] > threshold:
            # Tính khoảng cách
            dist = np.sqrt((filtered[p1, 0] - filtered[p2, 0])**2 + 
                          (filtered[p1, 1] - filtered[p2, 1])**2)
            
            # Nếu khoảng cách quá lớn, giảm confidence
            if dist > max_dist:
                penalty = 0.3 if is_symmetric else 0.5
                if filtered[p1, 2] < filtered[p2, 2]:
                    filtered[p1, 2] *= penalty
                else:
                    filtered[p2, 2] *= penalty
    
    # Kiểm tra keypoints nằm ngoài frame (y, x phải trong 0-1)
    for i in range(17):
        if filtered[i, 0] < 0 or filtered[i, 0] > 1 or \
           filtered[i, 1] < 0 or filtered[i, 1] > 1:
            filtered[i, 2] *= 0.1  # Giảm confidence mạnh
    
    return filtered


def draw_keypoints(frame, keypoints, width, height, threshold=0.3):
    """Vẽ các keypoints lên frame (keypoints format: y, x, conf)"""
    denormalized = np.squeeze(np.multiply(keypoints, [height, width, 1]))
    
    for i, keypoint in enumerate(denormalized):
        ky, kx, kconf = keypoint
        if kconf > threshold:
            # Màu dựa trên confidence: xanh lá (cao) -> đỏ (thấp)
            color_intensity = int(min(255, kconf * 255))
            color = (0, color_intensity, 255 - color_intensity)  # BGR
            
            cv2.circle(
                img=frame, 
                center=(int(kx), int(ky)), 
                radius=6, 
                color=color,
                thickness=-1
            )
    return denormalized


def draw_edges(denormalized, frame, edges_colors, threshold=0.3):
    """Vẽ các edges giữa các keypoints"""
    for edge, color in edges_colors.items():
        p1, p2 = edge
        y1, x1, conf1 = denormalized[p1]
        y2, x2, conf2 = denormalized[p2]
        
        if (conf1 > threshold) and (conf2 > threshold):
            avg_conf = (conf1 + conf2) / 2
            thickness = max(2, int(avg_conf * 5))
            
            cv2.line(
                img=frame, 
                pt1=(int(x1), int(y1)),
                pt2=(int(x2), int(y2)), 
                color=color, 
                thickness=thickness, 
                lineType=cv2.LINE_AA
            )

## 6. STEP 1: Extract Keypoints

Trích xuất keypoints từ video và lưu vào file JSON + NPY

**Output tensor format:** `(T, 17, [x, y, c])`

In [6]:
def extract_keypoints(input_video_path, keypoints_output_path,
                      threshold=0.3,
                      min_keypoints=6,
                      enable_smoothing=True,
                      smoothing_factor=0.5):
    """
    BƯỚC 1: Trích xuất keypoints từ video và lưu vào file
    
    Args:
        input_video_path: Đường dẫn video input
        keypoints_output_path: Đường dẫn file output (JSON)
        threshold: Ngưỡng confidence (0.25-0.4 recommended)
        min_keypoints: Số keypoints tối thiểu (6-10 recommended)
        enable_smoothing: Bật/tắt temporal smoothing
        smoothing_factor: Độ mượt (0.3-0.7 recommended)
    
    Returns:
        keypoints_data: Dict chứa metadata và keypoints
        
    Output:
        - JSON file với keypoints chi tiết
        - NPY tensor với shape (T, 17, 3) format [x, y, confidence]
    """
    
    video = cv2.VideoCapture(input_video_path)
    
    if not video.isOpened():
        print(f"❌ Error: Cannot open video file {input_video_path}")
        return None
    
    fps = int(video.get(cv2.CAP_PROP_FPS))
    frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    
    print("=" * 60)
    print("STEP 1: EXTRACTING KEYPOINTS")
    print("=" * 60)
    print(f"📁 Input: {input_video_path}")
    print(f"📐 Resolution: {frame_width}x{frame_height}")
    print(f"🎬 FPS: {fps}")
    print(f"📊 Total frames: {frame_count}")
    print(f"⏱️  Duration: {duration:.2f} seconds")
    print()
    print("PROCESSING SETTINGS")
    print("-" * 60)
    print(f"🤖 Model: MoveNet Singlepose Thunder")
    print(f"🎯 Confidence threshold: {threshold}")
    print(f"📍 Min keypoints required: {min_keypoints}")
    print(f"🔄 Temporal smoothing: {'ON' if enable_smoothing else 'OFF'}")
    if enable_smoothing:
        print(f"   Smoothing factor: {smoothing_factor}")
    print(f"📦 Output tensor format: (T, 17, [x, y, c])")
    print()
    
    # Khởi tạo smoother
    smoother = KeypointSmoother(smoothing_factor=smoothing_factor) if enable_smoothing else None
    
    # Khởi tạo data structure
    keypoints_data = {
        "metadata": {
            "video_file": input_video_path,
            "model": "MoveNet Singlepose Thunder",
            "fps": fps,
            "frame_width": frame_width,
            "frame_height": frame_height,
            "total_frames": frame_count,
            "duration_seconds": duration,
            "threshold": threshold,
            "min_keypoints": min_keypoints,
            "smoothing_enabled": enable_smoothing,
            "smoothing_factor": smoothing_factor if enable_smoothing else None,
            "keypoint_names": KEYPOINT_NAMES,
            "tensor_format": "(T, 17, [x, y, confidence])"
        },
        "frames": []
    }
    
    all_keypoints_raw = []  # Lưu raw numpy array với format (x, y, c)
    valid_frames = 0
    
    print("Extracting keypoints from video...")
    print("-" * 60)
    
    for _ in tqdm(range(frame_count), desc="Extracting", unit="frame"):
        ret, frame = video.read()
        
        if not ret:
            break
        
        frame_idx = len(all_keypoints_raw)
        
        # Resize và chuẩn bị input cho model
        input_image = tf.image.resize_with_pad(frame, WIDTH, HEIGHT)
        input_image = tf.cast(input_image, dtype=tf.int32)
        input_image = tf.expand_dims(input_image, axis=0)
        
        # Inference
        results = movenet(input_image)
        
        # Output shape: [1, 1, 17, 3] -> [17, 3] với format (y, x, confidence)
        keypoints = results["output_0"].numpy().reshape((17, 3))
        
        # Lọc outliers
        filtered_kp = filter_outlier_keypoints(keypoints, threshold)
        
        # Smooth
        if smoother:
            filtered_kp = smoother.smooth(filtered_kp)
        
        # Lưu raw keypoints với format (x, y, confidence)
        # Chuyển từ (y, x, conf) sang (x, y, conf)
        kp_reordered = filtered_kp[:, [1, 0, 2]]  # Swap y,x -> x,y
        all_keypoints_raw.append(kp_reordered)
        
        # Kiểm tra và tạo frame data
        is_valid = is_valid_skeleton(filtered_kp, threshold, min_keypoints)
        if is_valid:
            valid_frames += 1
        
        frame_data = {
            "frame_index": frame_idx,
            "timestamp_seconds": frame_idx / fps,
            "is_valid": bool(is_valid),
            "keypoints": {}
        }
        
        for kp_idx, kp_name in enumerate(KEYPOINT_NAMES):
            y, x, conf = filtered_kp[kp_idx]
            frame_data["keypoints"][kp_name] = {
                "x": float(x),
                "y": float(y),
                "confidence": float(conf),
                "pixel_x": float(x * frame_width),
                "pixel_y": float(y * frame_height)
            }
        
        keypoints_data["frames"].append(frame_data)
    
    video.release()
    
    # Cập nhật metadata với số valid frames
    keypoints_data["metadata"]["valid_frames"] = valid_frames
    
    # Lưu file JSON
    print()
    print("Saving keypoints data...")
    
    with open(keypoints_output_path, 'w', encoding='utf-8') as f:
        json.dump(keypoints_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Keypoints saved: {keypoints_output_path}")
    
    # Lưu file NPY (numpy binary - nhanh hơn để load)
    npy_path = keypoints_output_path.replace('.json', '.npy')
    np.save(npy_path, np.array(all_keypoints_raw))
    print(f"✅ NPY tensor saved: {npy_path}")
    print(f"   Shape: {np.array(all_keypoints_raw).shape}  (T, 17, [x,y,c])")
    
    print()
    print("=" * 60)
    print(f"📊 Valid skeleton frames: {valid_frames}/{frame_count} ({100*valid_frames/frame_count:.1f}%)")
    print("=" * 60)
    
    return keypoints_data

## 7. STEP 2: Render Video

Parse keypoints từ file JSON và vẽ skeleton lên video

In [7]:
def render_video(keypoints_json_path, input_video_path, output_video_path, threshold=0.3, min_keypoints=6):
    """
    BƯỚC 2: Parse keypoints từ file JSON và vẽ lên video
    
    Args:
        keypoints_json_path: Đường dẫn file keypoints JSON
        input_video_path: Đường dẫn video gốc
        output_video_path: Đường dẫn video output
        threshold: Ngưỡng confidence để vẽ
        min_keypoints: Số keypoints tối thiểu
    
    Returns:
        bool: True nếu thành công
    """
    
    # Load keypoints data
    print("=" * 60)
    print("STEP 2: RENDERING VIDEO")
    print("=" * 60)
    print(f"📄 Loading keypoints: {keypoints_json_path}")
    
    with open(keypoints_json_path, 'r', encoding='utf-8') as f:
        keypoints_data = json.load(f)
    
    metadata = keypoints_data["metadata"]
    frames_data = keypoints_data["frames"]
    
    print(f"   Model: {metadata['model']}")
    print(f"   Total frames: {metadata['total_frames']}")
    print(f"   Valid frames: {metadata['valid_frames']}")
    print()
    
    # Open video
    video = cv2.VideoCapture(input_video_path)
    
    if not video.isOpened():
        print(f"❌ Error: Cannot open video file {input_video_path}")
        return False
    
    fps = int(video.get(cv2.CAP_PROP_FPS))
    frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"📹 Video: {input_video_path}")
    print(f"📐 Resolution: {frame_width}x{frame_height}")
    print(f"🎬 FPS: {fps}")
    print()
    
    # Setup video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    
    print("Rendering skeleton onto video...")
    print("-" * 60)
    
    rendered_frames = 0
    frame_idx = 0
    
    for _ in tqdm(range(frame_count), desc="Rendering", unit="frame"):
        ret, frame = video.read()
        
        if not ret:
            break
        
        if frame_idx < len(frames_data):
            frame_data = frames_data[frame_idx]
            
            # Chuyển đổi keypoints từ dict sang numpy array (y, x, conf format cho drawing)
            keypoints = np.zeros((17, 3), dtype=np.float32)
            for kp_idx, kp_name in enumerate(KEYPOINT_NAMES):
                kp = frame_data["keypoints"][kp_name]
                keypoints[kp_idx] = [kp["y"], kp["x"], kp["confidence"]]
            
            # Kiểm tra và vẽ skeleton
            if is_valid_skeleton(keypoints, threshold, min_keypoints):
                denorm = draw_keypoints(frame, keypoints, frame_width, frame_height, threshold)
                draw_edges(denorm, frame, EDGE_COLORS, threshold)
                rendered_frames += 1
        
        out.write(frame)
        frame_idx += 1
    
    video.release()
    out.release()
    
    print()
    print("=" * 60)
    print("✅ RENDERING COMPLETED!")
    print("=" * 60)
    print(f"📹 Output video: {output_video_path}")
    print(f"📊 Rendered frames: {rendered_frames}/{frame_count}")
    print()
    
    return True

## 8. Combined Pipeline

Chạy cả 2 bước liên tiếp

In [8]:
def process_video_pipeline(input_video, output_video, keypoints_json,
                           threshold=0.3, min_keypoints=6,
                           enable_smoothing=True, smoothing_factor=0.5):
    """
    Pipeline đầy đủ: Extract keypoints -> Render video
    
    Args:
        input_video: Đường dẫn video input
        output_video: Đường dẫn video output
        keypoints_json: Đường dẫn file keypoints JSON
        threshold: Ngưỡng confidence (0.25-0.4 recommended)
        min_keypoints: Số keypoints tối thiểu (6-10 recommended)
        enable_smoothing: Bật temporal smoothing
        smoothing_factor: Độ mượt (0.3-0.7 recommended)
    """
    
    print()
    print("╔" + "═" * 58 + "╗")
    print("║" + " POSE ESTIMATION PIPELINE ".center(58) + "║")
    print("║" + " MoveNet Singlepose Thunder ".center(58) + "║")
    print("╚" + "═" * 58 + "╝")
    print()
    
    # Step 1: Extract keypoints
    keypoints_data = extract_keypoints(
        input_video_path=input_video,
        keypoints_output_path=keypoints_json,
        threshold=threshold,
        min_keypoints=min_keypoints,
        enable_smoothing=enable_smoothing,
        smoothing_factor=smoothing_factor
    )
    
    if keypoints_data is None:
        print("❌ Pipeline failed at Step 1")
        return False
    
    print()
    
    # Step 2: Render video
    success = render_video(
        keypoints_json_path=keypoints_json,
        input_video_path=input_video,
        output_video_path=output_video,
        threshold=threshold,
        min_keypoints=min_keypoints
    )
    
    if not success:
        print("❌ Pipeline failed at Step 2")
        return False
    
    print()
    print("╔" + "═" * 58 + "╗")
    print("║" + " ✅ PIPELINE COMPLETED SUCCESSFULLY! ".center(58) + "║")
    print("╚" + "═" * 58 + "╝")
    print()
    
    return True

## 9. Run Pipeline

### Cấu hình

| Parameter | Recommended | Description |
|-----------|-------------|-------------|
| `THRESHOLD` | 0.25-0.4 | Cao hơn = strict hơn, ít noise |
| `MIN_KEYPOINTS` | 6-10 | Số keypoints tối thiểu để skeleton valid |
| `SMOOTHING_FACTOR` | 0.3-0.7 | Cao hơn = mượt hơn nhưng lag hơn |

In [9]:
# === INPUT/OUTPUT FILES ===
INPUT_VIDEO = "Ha.mp4"
OUTPUT_VIDEO = "output_skeleton.mp4"
KEYPOINTS_JSON = "keypoints_data.json"

# === CẤU HÌNH (Improved) ===
THRESHOLD = 0.3            # Ngưỡng confidence (0.25-0.4 recommended)
MIN_KEYPOINTS = 6          # Ít nhất 6 keypoints + 2 core keypoints
ENABLE_SMOOTHING = True    # Bật temporal smoothing
SMOOTHING_FACTOR = 0.5     # Độ mượt (0.3-0.7 recommended)

### Option 1: Chạy cả Pipeline

In [10]:
process_video_pipeline(
    input_video=INPUT_VIDEO,
    output_video=OUTPUT_VIDEO,
    keypoints_json=KEYPOINTS_JSON,
    threshold=THRESHOLD,
    min_keypoints=MIN_KEYPOINTS,
    enable_smoothing=ENABLE_SMOOTHING,
    smoothing_factor=SMOOTHING_FACTOR
)


╔══════════════════════════════════════════════════════════╗
║                 POSE ESTIMATION PIPELINE                 ║
║                MoveNet Singlepose Thunder                ║
╚══════════════════════════════════════════════════════════╝

STEP 1: EXTRACTING KEYPOINTS
📁 Input: Ha.mp4
📐 Resolution: 1024x576
🎬 FPS: 30
📊 Total frames: 1336
⏱️  Duration: 44.53 seconds

PROCESSING SETTINGS
------------------------------------------------------------
🤖 Model: MoveNet Singlepose Thunder
🎯 Confidence threshold: 0.3
📍 Min keypoints required: 6
🔄 Temporal smoothing: ON
   Smoothing factor: 0.5
📦 Output tensor format: (T, 17, [x, y, c])

Extracting keypoints from video...
------------------------------------------------------------


Extracting:   0%|          | 0/1336 [00:00<?, ?frame/s]


Saving keypoints data...
✅ Keypoints saved: keypoints_data.json
✅ NPY tensor saved: keypoints_data.npy
   Shape: (1336, 17, 3)  (T, 17, [x,y,c])

📊 Valid skeleton frames: 1020/1336 (76.3%)

STEP 2: RENDERING VIDEO
📄 Loading keypoints: keypoints_data.json
   Model: MoveNet Singlepose Thunder
   Total frames: 1336
   Valid frames: 1020

📹 Video: Ha.mp4
📐 Resolution: 1024x576
🎬 FPS: 30

Rendering skeleton onto video...
------------------------------------------------------------


Rendering:   0%|          | 0/1336 [00:00<?, ?frame/s]


✅ RENDERING COMPLETED!
📹 Output video: output_skeleton.mp4
📊 Rendered frames: 1020/1336


╔══════════════════════════════════════════════════════════╗
║            ✅ PIPELINE COMPLETED SUCCESSFULLY!            ║
╚══════════════════════════════════════════════════════════╝



True

### Option 2: Chỉ Extract Keypoints

In [11]:
# extract_keypoints(
#     input_video_path=INPUT_VIDEO,
#     keypoints_output_path=KEYPOINTS_JSON,
#     threshold=THRESHOLD,
#     min_keypoints=MIN_KEYPOINTS,
#     enable_smoothing=ENABLE_SMOOTHING,
#     smoothing_factor=SMOOTHING_FACTOR
# )

### Option 3: Chỉ Render Video từ Keypoints đã có

In [12]:
# render_video(
#     keypoints_json_path=KEYPOINTS_JSON,
#     input_video_path=INPUT_VIDEO,
#     output_video_path=OUTPUT_VIDEO,
#     threshold=THRESHOLD,
#     min_keypoints=MIN_KEYPOINTS
# )

## 10. Load và Kiểm tra Tensor đã lưu

In [13]:
# Load tensor đã lưu
keypoints_tensor = np.load("keypoints_data.npy")

print(f"Tensor shape: {keypoints_tensor.shape}")
print(f"Format: (T, 17, [x, y, c])")
print()
print(f"T (số frames): {keypoints_tensor.shape[0]}")
print(f"17 keypoints per frame")
print(f"3 values per keypoint: [x, y, confidence]")
print()

# Xem keypoint đầu tiên của frame đầu tiên
print("Frame 0, Keypoint 0 (nose):")
print(f"  x: {keypoints_tensor[0, 0, 0]:.4f}")
print(f"  y: {keypoints_tensor[0, 0, 1]:.4f}")
print(f"  confidence: {keypoints_tensor[0, 0, 2]:.4f}")

Tensor shape: (1336, 17, 3)
Format: (T, 17, [x, y, c])

T (số frames): 1336
17 keypoints per frame
3 values per keypoint: [x, y, confidence]

Frame 0, Keypoint 0 (nose):
  x: 0.2513
  y: 0.3501
  confidence: 0.0634
